# Task 2 — Model Building & Training: Fraud_Data
**Adey Innovations Inc. — Fraud Detection Project**

This notebook covers:
1. Load processed data from Task 1
2. Stratified train-test split + SMOTE (training only)
3. Baseline model — Logistic Regression
4. Ensemble model — Random Forest
5. Stratified 5-Fold cross-validation
6. Model comparison & selection
7. Save models for Task 3

In [11]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from preprocess import split_and_resample
from modeling import (
    train_logistic_regression, train_random_forest,
    evaluate_model, plot_confusion_matrix, plot_precision_recall_curve,
    cross_validate_model, build_comparison_table, save_model
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Processed Data

In [2]:
fraud = pd.read_csv('../data/processed/fraud_processed.csv')
print('Shape:', fraud.shape)
fraud.head()

Shape: (129146, 199)


,purchase_value,age,class,time_since_signup,hour_of_day,day_of_week,user_tx_count,device_tx_count,source_Ads,source_Direct,...,country_United States,country_Uruguay,country_Uzbekistan,country_Vanuatu,country_Venezuela,country_Viet Nam,country_Virgin Islands (U.S.),country_Yemen,country_Zambia,country_Zimbabwe
0,0.549607,-0.363124,0,-0.413800,-1.231124,1.487911,0.0,-0.259874,False,False,...,False,False,False,False,False,False,False,False,False,False
1,-1.197335,0.101168,0,-1.180852,1.229002,-0.505034,0.0,-0.259874,False,False,...,False,False,False,False,False,False,False,False,False,False
2,0.385831,-0.479197,0,-0.936126,1.663142,0.989675,0.0,0.116936,True,False,...,False,False,False,False,False,False,False,False,False,False
3,0.986342,-0.363124,0,0.867086,0.650149,0.989675,0.0,-0.259874,False,True,...,False,False,False,False,False,False,False,False,False,False
4,0.767974,0.449387,0,1.700633,-1.086411,-1.003270,0.0,-0.259874,False,False,...,False,False,False,False,False,False,False,False,False,False


## 2. Stratified Train-Test Split + SMOTE

In [3]:
# split_and_resample:
#  - Splits data with stratify=y so fraud cases are proportionally represented in both sets
#  - Applies SMOTE ONLY on the training set
#  - Test set keeps the real-world class distribution for honest evaluation
X_train, X_test, y_train, y_test = split_and_resample(fraud, target_col='class')

print('X_train:', X_train.shape, '  y_train balance:', y_train.value_counts().to_dict())
print('X_test :', X_test.shape, '  y_test balance :', y_test.value_counts().to_dict())

INFO: Train size: 103316, Test size: 25830
INFO: Class distribution before SMOTE:
class
0    93502
1     9814
Name: count, dtype: int64
INFO: Class distribution after SMOTE:
class
0    93502
1    93502
Name: count, dtype: int64


X_train: (187004, 198)   y_train balance: {0: 93502, 1: 93502}
X_test : (25830, 198)   y_test balance : {0: 23376, 1: 2454}


## 3. Baseline Model — Logistic Regression

In [4]:
log_reg = train_logistic_regression(X_train, y_train)

lr_results = evaluate_model(log_reg, X_test, y_test, model_name='Logistic Regression')
lr_results['y_test'] = y_test

plot_confusion_matrix(lr_results['confusion_matrix'], model_name='Logistic Regression', dataset_name='Fraud Data')

INFO: Logistic Regression trained.
INFO: Logistic Regression -> AUC-PR: 0.6376, F1: 0.6842
INFO: Confusion Matrix:
[[23205   171]
 [ 1089  1365]]


  Saved → outputs/confusion_matrix_logistic_regression_fraud_data.png


> **Why Logistic Regression as baseline:** It's fast, interpretable (coefficients show feature direction/strength), and gives a reference point to judge whether the added complexity of an ensemble model is worth it.

## 4. Ensemble Model — Random Forest

In [5]:
rf = train_random_forest(X_train, y_train, n_estimators=100, max_depth=10)

rf_results = evaluate_model(rf, X_test, y_test, model_name='Random Forest')
rf_results['y_test'] = y_test

plot_confusion_matrix(rf_results['confusion_matrix'], model_name='Random Forest', dataset_name='Fraud Data')

INFO: Random Forest trained (n_estimators=100, max_depth=10).
INFO: Random Forest -> AUC-PR: 0.7191, F1: 0.6257
INFO: Confusion Matrix:
[[22026  1350]
 [  722  1732]]


  Saved → outputs/confusion_matrix_random_forest_fraud_data.png


In [6]:
# Precision-Recall curve comparing both models
plot_precision_recall_curve([lr_results, rf_results], dataset_name='Fraud Data')

  Saved → outputs/precision_recall_fraud_data.png


## 5. Stratified 5-Fold Cross-Validation

In [7]:
# Cross-validate on the SMOTE-resampled training data
# A fresh model is trained on each fold to avoid data leakage

print('Cross-validating Logistic Regression...')
cv_lr = cross_validate_model(
    lambda: LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
    X_train, y_train, k=5
)

print('\nCross-validating Random Forest...')
cv_rf = cross_validate_model(
    lambda: RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    X_train, y_train, k=5
)

Cross-validating Logistic Regression...


INFO: Fold 1: AUC-PR=0.9554, F1=0.8750
INFO: Fold 2: AUC-PR=0.9556, F1=0.8737
INFO: Fold 3: AUC-PR=0.9553, F1=0.8720
INFO: Fold 4: AUC-PR=0.9551, F1=0.8757
INFO: Fold 5: AUC-PR=0.9544, F1=0.8707



Cross-validating Random Forest...


INFO: Fold 1: AUC-PR=0.9592, F1=0.8379
INFO: Fold 2: AUC-PR=0.9563, F1=0.8462
INFO: Fold 3: AUC-PR=0.9564, F1=0.8444
INFO: Fold 4: AUC-PR=0.9566, F1=0.8405
INFO: Fold 5: AUC-PR=0.9541, F1=0.8419


In [8]:
# Attach CV results to the evaluation dicts
lr_results.update({
    'cv_auc_pr_mean': cv_lr['auc_pr_mean'], 'cv_auc_pr_std': cv_lr['auc_pr_std'],
    'cv_f1_mean': cv_lr['f1_mean'], 'cv_f1_std': cv_lr['f1_std']
})
rf_results.update({
    'cv_auc_pr_mean': cv_rf['auc_pr_mean'], 'cv_auc_pr_std': cv_rf['auc_pr_std'],
    'cv_f1_mean': cv_rf['f1_mean'], 'cv_f1_std': cv_rf['f1_std']
})

## 6. Model Comparison & Selection

In [9]:
comparison = build_comparison_table([lr_results, rf_results])
comparison

,Model,Test AUC-PR,Test F1-Score,CV AUC-PR (mean ± std),CV F1 (mean ± std)
0,Logistic Regression,0.6376,0.6842,0.9552 ± 0.0004,0.8734 ± 0.0019
1,Random Forest,0.7191,0.6257,0.9565 ± 0.0016,0.8422 ± 0.0029


> **Model Selection Justification:**
>
> The Random Forest model is expected to outperform Logistic Regression on both AUC-PR and F1-Score, because:
> - It captures non-linear relationships and feature interactions (e.g., between `time_since_signup` and `device_tx_count`)
> - Tree-based models handle the mix of one-hot encoded categorical features and scaled numeric features well without needing manual interaction terms
> - Logistic Regression assumes a linear decision boundary, which is unlikely to capture fraud patterns that depend on combinations of behavioural signals
>
> **Trade-off:** Logistic Regression remains useful as an interpretable baseline and for quick re-training. Random Forest is selected as the **primary model** for Task 3 SHAP analysis due to its superior predictive performance, while Logistic Regression coefficients can still be referenced as a sanity check for feature direction.
>
> *(Update this paragraph with the actual numeric comparison once you've run the cells above.)*

## 7. Save Models

In [10]:
import os
os.makedirs('../data/models', exist_ok=True)

# Save from notebooks dir, so models/ goes to ../data/models
import joblib
joblib.dump(log_reg, '../data/models/logistic_regression_fraud_data.joblib')
joblib.dump(rf, '../data/models/random_forest_fraud_data.joblib')

# Also save the test set for Task 3 SHAP analysis
X_test.to_csv('../data/processed/fraud_X_test.csv', index=False)
y_test.to_csv('../data/processed/fraud_y_test.csv', index=False)

print('✅ Models and test set saved.')

✅ Models and test set saved.


## Summary

| Model | Test AUC-PR | Test F1 | CV AUC-PR | CV F1 |
|-------|-------------|---------|-----------|-------|
| Logistic Regression | see table above | | | |
| Random Forest | see table above | | | |

**Selected model for Task 3:** Random Forest (pending confirmation from comparison table)